# PR open durations


## Preliminaries


In [ ]:
# Run me to download latest data from GitHub

from qb_notebook.artifacts import (
    download_and_extract_latest_successful_workflow_artifacts,
)

info = download_and_extract_latest_successful_workflow_artifacts(
    repo="leanprover-community/queueboard-core",
    workflow="upload_backup.yaml",
    out_dir="./data",
    artifact_name="analytics-datasets",
    branch="master",
    search_limit=100,  # change this if you expect there to be > 100 failed runs before the first successful one
)

info

In [ ]:
import polars as pl

from qb_notebook.data_io import load_pr_interval_data

tables = load_pr_interval_data("data")

df_prs = tables["prs"]
df_events = tables["events"]
df_label_defs = tables["label_defs"]
df_prlabel = tables["prlabel"]

## Open intervals

In [ ]:
from qb_notebook.intervals import build_pr_open_intervals

In [ ]:
df_intervals = build_pr_open_intervals(df_prs, df_events)

# Or fix as-of time for reproducibility:
# df_intervals_pl = build_pr_open_intervals(
#     df_prs, df_events,
#     asof=pl.lit(pl.datetime(2025, 12, 17, 0, 0, 0, time_unit="us"))
# )

## Filtering Classes of Intervals

In [ ]:
from qb_notebook.intervals import enrich_intervals_with_prs

In [ ]:
df_intervals_enriched = enrich_intervals_with_prs(df_intervals, df_prs)

In [ ]:
from qb_notebook.filters import (
    expr_interval_started_between,
    expr_only_closed,
    expr_pr_has_any_of,
    expr_title_regex,
    filter_rows,
    pr_ids_with_any_labels,
)

In [ ]:
df_closed = filter_rows(df_intervals_enriched, expr_only_closed(True))
print("df_closed", len(df_closed))

df_open = filter_rows(df_intervals_enriched, expr_only_closed(False))
print("df_open", len(df_open))
df_open_feat = filter_rows(df_open, expr_title_regex(r"^feat"))
print("df_open_feat", len(df_open_feat))

# doesn't count PRs that actually got merged
df_merged = filter_rows(df_closed, expr_title_regex(r"^\[Merged by Bors\] -"))
print("df_merged", len(df_merged))
df_merged_feat = filter_rows(
    df_closed, expr_title_regex(r"^\[Merged by Bors\] -\s+[fF]eat")
)
print("df_merged_feat", len(df_merged_feat))
df_merged_nonfeat = filter_rows(
    df_closed, expr_title_regex(r"^\[Merged by Bors\] -\s+[^\sfF]")
)
print("df_merged_nonfeat", len(df_merged_nonfeat))

In [ ]:
df_merged_2021 = filter_rows(
    df_merged,
    expr_interval_started_between(start_after="2021-01-01", start_before="2022-01-01"),
)
print("df_merged_2021", len(df_merged_2021))
df_merged_2022 = filter_rows(
    df_merged,
    expr_interval_started_between(start_after="2022-01-01", start_before="2023-01-01"),
)
print("df_merged_2022", len(df_merged_2022))
df_merged_2023 = filter_rows(
    df_merged,
    expr_interval_started_between(start_after="2023-01-01", start_before="2024-01-01"),
)
print("df_merged_2023", len(df_merged_2023))
df_merged_2024 = filter_rows(
    df_merged,
    expr_interval_started_between(start_after="2024-01-01", start_before="2025-01-01"),
)
print("df_merged_2024", len(df_merged_2024))
df_merged_2025 = filter_rows(
    df_merged,
    expr_interval_started_between(start_after="2025-01-01", start_before="2026-01-01"),
)
print("df_merged_2025", len(df_merged_2025))
df_merged_2026 = filter_rows(
    df_merged,
    expr_interval_started_between(start_after="2026-01-01", start_before="2027-01-01"),
)
print("df_merged_2026", len(df_merged_2026))

In [ ]:
df_merged_feat_after_port = filter_rows(
    df_merged_feat, expr_interval_started_between(start_after="2023-08-01")
)
print("df_merged_feat_after_port", len(df_merged_feat_after_port))
df_merged_nonfeat_after_port = filter_rows(
    df_merged_nonfeat, expr_interval_started_between(start_after="2023-08-01")
)
print("df_merged_nonfeat_after_port", len(df_merged_nonfeat_after_port))

In [ ]:
ids_new_contributor = pr_ids_with_any_labels(
    df_prlabel, df_label_defs, ["new-contributor"]
)

df_merged_new_contributor = filter_rows(
    df_merged,
    expr_pr_has_any_of(ids_new_contributor),
)
print("df_merged_new_contributor", len(df_merged_new_contributor))

## Open PRs vs time

In [ ]:
from qb_notebook.intervals import effective_open_prs_per_day

In [ ]:
import altair as alt

daily_open = effective_open_prs_per_day(df_intervals)
daily_open.plot.line(x="date", y="open_prs")

## PRs merged per day

In [ ]:
daily_closed = (
    df_merged.filter(pl.col("closed_at").is_not_null())
    .with_columns(pl.col("closed_at").dt.truncate("1d").alias("date"))
    .group_by("date")
    .agg(pl.len().alias("prs_closed"))
    .sort("date")
)

daily_closed_smoothed = daily_closed.with_columns(
    pl.col("prs_closed").rolling_mean(14).alias("prs_closed_14d_avg")
)

daily_closed_smoothed.plot.line(x="date", y="prs_closed_14d_avg")

In [ ]:
daily_closed_nc = (
    df_merged_new_contributor.filter(pl.col("closed_at").is_not_null())
    .with_columns(pl.col("closed_at").dt.truncate("1d").alias("date"))
    .group_by("date")
    .agg(pl.len().alias("prs_closed"))
    .sort("date")
)

daily_closed_nc_smoothed = daily_closed_nc.with_columns(
    pl.col("prs_closed").rolling_mean(14).alias("prs_closed_14d_avg")
)

daily_closed_nc_smoothed.plot.line(x="date", y="prs_closed_14d_avg")

In [ ]:
nc_ratio = (
    daily_closed.select(["date", pl.col("prs_closed").alias("prs_closed_all")])
    .join(
        daily_closed_nc.select(["date", pl.col("prs_closed").alias("prs_closed_nc")]),
        on="date",
        how="inner",
    )
    .with_columns(
        (pl.col("prs_closed_nc") / pl.col("prs_closed_all")).alias("nc_ratio")
    )
    .select(["date", "nc_ratio"])
    .sort(
        "date"
    )  # must sort before rolling_mean; Polars rolling_mean is row-order-based, not time-based
)

nc_ratio_smoothed = nc_ratio.with_columns(
    pl.col("nc_ratio").rolling_mean(14).alias("nc_ratio_14d_avg")
)

alt.Chart(nc_ratio_smoothed.to_pandas()).mark_line().encode(
    x="date:T",
    y="nc_ratio_14d_avg:Q",
).interactive()

In [ ]:
nc_smoothed_ratio = (
    daily_closed_smoothed.select(
        ["date", pl.col("prs_closed_14d_avg").alias("prs_closed_all")]
    )
    .join(
        daily_closed_nc_smoothed.select(
            ["date", pl.col("prs_closed_14d_avg").alias("prs_closed_nc")]
        ),
        on="date",
        how="inner",
    )
    .with_columns(
        (pl.col("prs_closed_nc") / pl.col("prs_closed_all")).alias("nc_smoothed_ratio")
    )
    .select(["date", "nc_smoothed_ratio"])
)

# nc_smoothed_ratio_smoothed = (
#     nc_smoothed_ratio
#     .with_columns(
#         pl.col("nc_ratio")
#         .rolling_mean(14)
#         .alias("nc_ratio_14d_avg")
#     )
# )

alt.Chart(nc_smoothed_ratio.to_pandas()).mark_line().encode(
    x="date:T",
    y="nc_smoothed_ratio:Q",
).interactive()

## Interval plot

In [ ]:
from qb_notebook.plotting import prepare_swimlane_polars, plot_swimlane_matplotlib

In [ ]:
swim_all = prepare_swimlane_polars(df_intervals)  # all PRs
plot_swimlane_matplotlib(swim_all, figsize=(5, 25), linewidth=0.4)

## Open Durations

### Helper functions

In [ ]:
import numpy as np

from qb_notebook.plotting import (
    plot_duration_hist,
    plot_duration_hists,
    plot_loglogistic_fit_counts_logbins,
    plot_lognormal_fit_counts_logbins,
    plot_weibull_fit_counts_logbins,
)

### Basic plots

In [ ]:
plot_duration_hist(df_closed, logx=True)

In [ ]:
plot_duration_hist(df_open, logx=True)

In [ ]:
plot_duration_hist(df_open, logy=True, exponential_fit=True)

In [ ]:
1 / 0.00417

In [ ]:
df_open["duration_days"].median()

In [ ]:
plot_duration_hist(df_merged, logx=True)

### Plots with fits

In [ ]:
merged_lognormal_params = plot_lognormal_fit_counts_logbins(df_merged)

In [ ]:
np.exp(merged_lognormal_params["mu"])

In [ ]:
df_merged["duration_days"].median()

In [ ]:
plot_loglogistic_fit_counts_logbins(df_merged)

In [ ]:
plot_weibull_fit_counts_logbins(df_merged)

### More lognormal plots

In [ ]:
plot_lognormal_fit_counts_logbins(df_merged, col="duration_days", bins=100)

In [ ]:
np.exp(0.454)

In [ ]:
df_merged["duration_days"].median()

In [ ]:
plot_lognormal_fit_counts_logbins(df_merged_2021, col="duration_days", bins=100)

In [ ]:
plot_lognormal_fit_counts_logbins(df_merged_2022, col="duration_days", bins=100)

In [ ]:
plot_lognormal_fit_counts_logbins(df_merged_2023, col="duration_days", bins=100)

In [ ]:
plot_lognormal_fit_counts_logbins(df_merged_2024, col="duration_days", bins=100)

In [ ]:
plot_lognormal_fit_counts_logbins(df_merged_2025, col="duration_days", bins=100)

### Combined plots

In [ ]:
from qb_notebook.plotting import plot_hist_and_lognormal_fit_overlays

In [ ]:
dfs = [
    df_merged_2021,
    df_merged_2022,
    df_merged_2023,
    df_merged_2024,
    df_merged_2025,
    df_merged_2026,
]
labels = ["2021", "2022", "2023", "2024", "2025", "2026"]

plot_hist_and_lognormal_fit_overlays(
    dfs,
    labels,
    col="duration_days",
    bins=100,
    title="Merged PR interval durations",
    show_params_in_legend=False,  # set True if you want μ/σ in legend
)

In [ ]:
print(
    len(df_merged_2021),
    len(df_merged_2022),
    len(df_merged_2023),
    len(df_merged_2024),
    len(df_merged_2025),
    len(df_merged_2026),
)

In [ ]:
dfs_feat = [df_merged_feat, df_merged_nonfeat]
labels = ["feat", "non-feat"]

plot_hist_and_lognormal_fit_overlays(
    dfs_feat,
    labels,
    col="duration_days",
    bins=100,
    title="Merged PR interval durations",
    show_params_in_legend=True,
)

In [ ]:
np.exp(1.14)

In [ ]:
np.exp(-0.32)

In [ ]:
dfs_feat_after_port = [df_merged_feat_after_port, df_merged_nonfeat_after_port]
labels = ["feat (after 2023-08-01)", "non-feat (after 2023-08-01)"]

plot_hist_and_lognormal_fit_overlays(
    dfs_feat_after_port,
    labels,
    col="duration_days",
    bins=100,
    title="Merged PR interval durations",
    show_params_in_legend=True,  # set True if you want μ/σ in legend
)

In [ ]:
np.exp(1.57)

In [ ]:
np.exp(-0.22)